In [1]:
!cp "/content/drive/MyDrive/Hate Speech LLMs/Data ICWSM/Eval Data/qwenInstData.xlsx" .

In [2]:
import numpy as np
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score

In [3]:
df = pd.read_excel("qwenInstData.xlsx")

In [4]:
ref_col = 'Hop_2_Target_Basis_of_Identity (Ref)'
pred_col = 'Hop_2_Target_Basis_of_Identity'

In [5]:
df = df[df["Hop_1_Triage (Ref)"] != "بی‌ضرر"]
df = df[df["Hop_1_Triage"] != "بی‌ضرر"]
print(len(df))

846


In [6]:
refIdentities = []
for refIdentity in df[ref_col]:
  refIdentities.extend(ast.literal_eval(refIdentity))
set(refIdentities)

{'other',
 'جنسیت',
 'حرفه/شغل',
 'دین/باور',
 'سوابق تحصیلی/دانشگاهی',
 'طبقه اجتماعی',
 'عضویت گروهی',
 'قومیت/ملیت',
 'ماهیت انسانی',
 'نقش اجتماعی/خانوادگی',
 'هویت فردی',
 'وابستگی سازمانی/نهادی',
 'وضعیت مهاجرت',
 'ویژگی ذهنی/رفتاری',
 'گرایش سیاسی'}

In [7]:
llmIdentities = []
for llmIdentity in df[pred_col]:
  llmIdentities.extend(ast.literal_eval(llmIdentity))
set(llmIdentities)

{'other',
 'جنسیت',
 'حرفه/شغل',
 'دین/باور',
 'سن',
 'سن/نوجوانی',
 'سن/وضعیت سنی',
 'سوابق تحصیلی/دانشگاهی',
 'طبقه اجتماعی',
 'عضویت گروهی',
 'عمر',
 'قومیت/ملیت',
 'مبنای سیاسی',
 'مذهب',
 'ملیت',
 'ملیت/قومیت',
 'موقعیت سیاسی',
 'نقش اجتماعی/خانوادگی',
 'هویت فردی',
 'وابستگی سازمانی/نهادی',
 'وضعیت مهاجرت',
 'وظیفه اجتماعی/سیاسی',
 'وظیفه سیاسی/نظامی',
 'وظیفه نظامی',
 'وظیفه/نقش اجتماعی',
 'وظیفه/نقش سیاسی',
 'ویژگی ذهنی/رفتاری',
 'گرایش جنسی',
 'گرایش سیاسی'}

In [8]:
len(set(refIdentities))

15

In [9]:
len(set(llmIdentities))

29

In [10]:
df[ref_col] = df[ref_col].apply(ast.literal_eval)
df[pred_col] = df[pred_col].apply(ast.literal_eval)

In [11]:
df[ref_col] = df[ref_col].apply(
    lambda x: [label for label in x if label != 'other']
)

df[pred_col] = df[pred_col].apply(
    lambda x: [label for label in x if label != 'other']
)

df_eval = df[
    df[ref_col].apply(len).gt(0) &
    df[pred_col].apply(len).gt(0)
].copy()

In [12]:
mlb = MultiLabelBinarizer()

mlb.fit(list(df_eval[ref_col]) + list(df_eval[pred_col]))

y_true = mlb.transform(df_eval[ref_col])
y_pred = mlb.transform(df_eval[pred_col])

In [13]:
f1_micro = f1_score(y_true, y_pred, average='micro')
f1_macro = f1_score(y_true, y_pred, average='macro')

print(f"Micro F1: {f1_micro:.4f}")
print(f"Macro F1: {f1_macro:.4f}")

Micro F1: 0.5273
Macro F1: 0.1461
